# Using APIs for Chemistry — full answers

**Aim:** retrieve chemical data, inspect JSON, build tables and compare information from PubChem and the Protein Data Bank (PDB).

An **API** (application programming interface) lets your code request information from a service. An **endpoint** is the web address used for a particular request. Here, we use Python's built-in `urllib` module.

Work from top to bottom. Read each explanation before running the following cell. All code blanks are completed here. Some cells provide complete examples; later tasks ask you to apply them. Write answers in the spaces provided. Outputs are generated when you run the code.
You need basic Python, internet access, NumPy, pandas, matplotlib and IPython/Jupyter. Install missing packages with `%pip install numpy pandas matplotlib ipython` in a new cell. 

The companion answer notebook contains completed code and suggested written answers. API results can change over time.

## Before you start

PubChem asks users to make no more than five requests per second. Use a 0.25-second pause between repeated requests, and pause longer if the server reports throttling. A **request** is the message you send; a **response** is what the server returns. **HTTP** is the web protocol used to exchange them.

We give requests a 30-second timeout so they cannot wait indefinitely. If a service is unavailable, stop and retry later rather than repeatedly sending requests.

Official documentation (checked 6 October 2026):
- [PubChem PUG REST](https://pubchem.ncbi.nlm.nih.gov/docs/pug-rest)
- [RCSB Search API](https://search.rcsb.org/)
- [pandas JSON documentation](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.to_json.html)

### Task 1 — Import the tools

In the code cell below import the following

- `urlopen`, `Request` from `urllib.request`: send web requests.
- `quote` from `urllib.parse`: make names safe to put in URLs.
- `HTTPError`, `URLError` from `urllib.error`: identify request failures.
- `Image`, `display` from `IPython.display`: show images and tables.
- `StringIO` from `io`: let pandas read text as if it were a file.
- `datetime`, `timezone` from `datetime`: record retrieval dates.
- `json`
- `time`
- `numpy` as `np`
- `matplotlib.pyplot` as `plt`, and 
- `pandas` as `pd`.

`urllib`, `json`, `time`, `io` and `datetime` are included with Python.

## Part 1 — Your first chemistry API request

A PubChem PUG REST request can often be read almost like a sentence.

For example, the following asks PubChem for the molecular formula of aspirin, returned as JSON:

`https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/name/aspirin/property/MolecularFormula/JSON`

The response is text transmitted over HTTP. Python must:

1. open the URL,
2. read the bytes returned by the server,
3. decode the bytes into text,
4. interpret the text as JSON.


### Task 2 — Open a URL and inspect the response

Complete the code to request the molecular formula of aspirin and print:

- the type returned directly by `.read()`
- the first 200 characters after decoding it as UTF-8



In [2]:
url = "https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/name/aspirin/property/MolecularFormula/JSON"
with urlopen(url, timeout=30) as response:
    raw_data = ______.read()
print(type(_____))
text = _____.decode("utf-8")
print(______[:200])

<class 'bytes'>
{
  "PropertyTable": {
    "Properties": [
      {
        "CID": 2244,
        "MolecularFormula": "C9H8O4"
      }
    ]
  }
}



### Task 3 — Read JSON and explore nesting

The response from PuBCHEM in in JSON format. JSON is a text format for organised data. `json.loads()` converts JSON text into Python objects. JSON objects `{}` become **dictionaries**; arrays `[]` become **lists**. Text becomes `str`, numbers become `int` or `float`, and `null` becomes `None`.

First convert the text from Task 2 into Python data using the commmand 

>json.loads

In [4]:
data = ____._____(_____)

#### Keys, lists and nesting

A **dictionary** uses `{}` and stores **key–value pairs** e.g.
```python
{"name": "ethanol"}
```
The key is the name `'name'`; its value is `'ethanol'`. Retrieve a value using its key: `example['name']`.

Remember: A **list** uses `[]` and stores items in order. Retrieve an item using its position, called an **index**: `example[0]` selects the first item. Python starts counting at zero.

**Nesting** means putting one dictionary or list inside another. You may need several selections to reach a value. The **top level** is the outermost container, which holds everything else.

For example:
```python
{"sample": {"measurements": [12.4, 13.1]}}
```
Here, a dictionary contains another dictionary, which contains a list. Work through one container at a time. Use names for dictionaries and positions for lists.

Inspect the data:
1. Use `type(data)` to identify the top-level container.
2. If it is a dictionary, use `.keys()` to display its keys.
3. Print `data` to see what those keys contain.



In [5]:
print(_____)

<class 'dict'>


In [6]:
print(____._____())

dict_keys(['PropertyTable'])


In [7]:
print(____)

{'PropertyTable': {'Properties': [{'CID': 2244, 'MolecularFormula': 'C9H8O4'}]}}


Choose the key containing the property information from your output. Store its value in `property_table`, then inspect it. Dictionary keys that are text must be in quotation marks.

In [8]:
property_table = data[______]
print(_______)

{'Properties': [{'CID': 2244, 'MolecularFormula': 'C9H8O4'}]}


**Explanation:** the top level is a dictionary. `PropertyTable` contains another dictionary; `Properties` contains a list of compound dictionaries. The molecular formula is the value under `MolecularFormula`. 

You can now take the column called "Properties" from the table property_table and store it in a new variable called properties. 
The first item in the list is the molecule ID, store this as a variable called molecule.
Finally store the 'MolecularFormular' of the molecule as `formula` and print it.


In [21]:
properties = _______[______]
molecule = properties[0]
print(______)
formula = molecule[________]
print(f'Molecular formula:', _______)

{'CID': 2244, 'MolecularFormula': 'C9H8O4'}
Molecular formula: C9H8O4


This process can be condensed into a single command

> asprin_formula = data["First Key"]["Second Key"][Index][Third key]

In [25]:
asprin_formula = data[_______][_______][0][_______]
print(f'Asprin formula:',________)

Asprin formula: C9H8O4


## Part 2 — Building API URLs safely

Chemical names are not always simple strings such as `aspirin`. They may contain spaces, brackets, `+`, `/`, commas, or other characters with special meaning in a URL.

`urllib.parse.quote()` converts a string into a URL-safe representation.

For example:

```python
quote("sodium chloride")
```

does not return exactly the same string.


### Task 4 — Write a reusable PubChem function

A **function** groups code so you can reuse it with different inputs. Write `get_pubchem_properties(name)` to encode the name, open the URL, request the listed properties, decode the response as utf-8 and return the first property dictionary.

Use `quote(name, safe="")`: this also encodes slashes that could otherwise change the URL path. We request `ConnectivitySMILES`, PubChem's current name for a SMILES representation without stereochemistry. A **SMILES** string represents molecular connectivity as text. PubChem automatically includes the compound identifier (**CID**) in property responses.



In [27]:
def ________(____):

    encoded_name = ____(_____, _____="")

    properties = (
        "MolecularFormula,MolecularWeight,XLogP,"
        "HBondDonorCount,HBondAcceptorCount,TPSA,ConnectivitySMILES"
    )

    url = (
        "https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/name/"
        + encoded_name
        + "/property/"
        + properties
        + "/JSON"
    )

    with ______(url, timeout=___) as response:
        text = ____.____().____(____)

    data = json.loads(text)



    compound = _____[____][_____][__] 
    return ________

#call the function for sodium chloride and print the output
Sodium_Chloride = _________("______")
print(________)


{'CID': 5234, 'MolecularFormula': 'ClNa', 'MolecularWeight': '58.44', 'ConnectivitySMILES': '[Na+].[Cl-]', 'TPSA': 0, 'HBondDonorCount': 0, 'HBondAcceptorCount': 1}


### Task 5 — Select useful information

Print a report for caffeine using dictionary keys:
- CID: PubChem compound identifier.
- Molecular formula.
- Molecular weight: numerical molar mass in g mol⁻¹.
- XLogP: a calculated estimate of log₁₀ of the octanol/water partition coefficient. Larger values generally indicate greater lipophilicity; this is not measured aqueous solubility.
- Hydrogen-bond donor and acceptor counts.
- TPSA: topological polar surface area in Å², estimated from the molecular graph.

Use `.get('XLogP')` for a property that might be missing. It returns `None` when the key is absent, rather than raising `KeyError`.

In [28]:
print('CID:', caffeine['CID'])
print(____)
print(____)
print(____)
print(____)
print(____)
print(____)

CID: 5234
Formula: ClNa
Molecular weight: 58.44
XLogP: None
H-bond donors: 0
H-bond acceptors: 1
TPSA: 0


## Part 3 — What happens when an API request fails?

A useful scientific program should not crash because one compound name is misspelled or a server is temporarily unavailable.

`urllib` raises exceptions such as:

- `HTTPError` — the server responded, but with an HTTP error status such as 404.
- `URLError` — Python could not successfully reach the server.

We can handle these using `try` / `except`.


### Task 6 — Handle failed requests

An **exception** is Python's signal that an operation failed. A `try` block attempts an operation; `except` handles a named failure.

Make a function to determine errors from get_pubchem_properties. Then call `safe_pubchem_properties(name)` using a print command an informative message if the request fails, and return `None` for a failure. Handle `HTTPError` (server error status), `URLError` (connection problem) and `TimeoutError`.

Test a real compound and a nonsense name. A failed request is missing information, not a molecular property of zero.

In [14]:
def safe_pubchem_properties(name):
    try:
        return get_pubchem_properties(name)
    except HTTPError as error:
        print('HTTP error for', ____, ':', error.code)
        return None
    except URLError as error:
        print('Connection error for', ____, ':', error.reason)
        return None
    except TimeoutError:
        print('Request timed out for', ____)
        return None

print(safe_pubchem_properties('_____'))
time.sleep(0.25)
print(safe_pubchem_properties('_______'))

{'CID': 1983, 'MolecularFormula': 'C8H9NO2', 'MolecularWeight': '151.16', 'ConnectivitySMILES': 'CC(=O)NC1=CC=C(C=C1)O', 'XLogP': 0.5, 'TPSA': 49.3, 'HBondDonorCount': 2, 'HBondAcceptorCount': 2}
HTTP error for this_is_not_a_real_compound_12345 : 404
None


## Part 4 — Automating a chemical comparison

Suppose we want to compare a set of common molecules:

```python
compounds = [
    "caffeine",
    "paracetamol",
    "aspirin",
    "ibuprofen",
    "naproxen",
    "nicotinamide",
    "benzene",
    "phenol"
]
```

An API allows us to retrieve the same set of properties reproducibly for every molecule.

When repeating requests, we should avoid overloading the server. A delay of 0.25 s corresponds to at most about four requests per second.


### Task 7 — Query several compounds

Loop over the list below.

For each compound:

1. call `safe_pubchem_properties()`,
2. if the result is not `None`, add the original search name under a new key called `"SearchName"`,
3. append the dictionary to `records`,
4. wait `0.25` seconds before the next request.

Print the number of successful records.

In [30]:
compounds = [
    "caffeine",
    "paracetamol",
    "aspirin",
    "ibuprofen",
    "naproxen",
    "nicotinamide",
    "benzene",
    "phenol"
]

records = []

for ____ in ______:

    properties = ______(____)

    if properties is not None:
        properties["______"] = name
        records.append(properties)

    time.sleep(0.25)

print("Successful records:", ___(____))


Successful records: 8


### Task 8 — Convert the API results into a table

Convert `records` into a DataFrame.

Display only:

- `SearchName`
- `CID`
- `MolecularFormula`
- `MolecularWeight`
- `XLogP`
- `HBondDonorCount`
- `HBondAcceptorCount`
- `TPSA`

The molecular weight values retrieved from the API are stored as strings (text) rather than numerical data, meaning they cannot be used directly in calculations. Use `pd.to_numeric()` to convert the MolecularWeight column into numbers, including `errors="coerce"` to replace any values that cannot be converted with NaN (Not a Number) rather than causing an error.

Then sort the table by molecular weight using df.sort_values



In [31]:


df = pd.DataFrame(records)

columns = [
    "SearchName",
    "CID",
    "MolecularFormula",
    "MolecularWeight",
    "XLogP",
    "HBondDonorCount",
    "HBondAcceptorCount",
    "TPSA"
]

if df.empty:
    raise RuntimeError("No compound records were retrieved; check requests before continuing.")
df = df[columns]

#convert molecular weight  to numeric values

df["____"] = pd.to_numeric(df["____"], errors="____")

#sort by molecular weight

df = ___.____(
    by="____"
)

df


,SearchName,CID,MolecularFormula,MolecularWeight,XLogP,HBondDonorCount,HBondAcceptorCount,TPSA
6,benzene,241,C6H6,78.11,2.1,0,0,0.0
7,phenol,996,C6H6O,94.11,1.5,1,1,20.2
5,nicotinamide,936,C6H6N2O,122.12,-0.4,1,2,56.0
1,paracetamol,1983,C8H9NO2,151.16,0.5,2,2,49.3
2,aspirin,2244,C9H8O4,180.16,1.2,1,4,63.6
0,caffeine,2519,C8H10N4O2,194.19,-0.1,0,3,58.4
3,ibuprofen,3672,C13H18O2,206.28,3.5,1,2,37.3
4,naproxen,156391,C14H14O3,230.26,3.3,1,3,46.5


**Suggested answer:** A table puts one compound per row and one property per column, making sorting, filtering and plotting easier than repeatedly navigating nested JSON.

### JSON and pandas — the `orient` argument

You have now converted API results into a pandas DataFrame. We can also go in the other direction and represent a DataFrame as JSON.

A DataFrame has rows and columns, whereas JSON is organised using objects and arrays. The pandas `orient` argument controls how the rows and columns are arranged in the JSON output.

We will compare three orientations:

- `orient="records"` — each row becomes an object in a JSON list;
- `orient="columns"` — columns form the outer structure;
- `orient="index"` — DataFrame index values form the outer structure.

Start with this illustrative DataFrame. These numbers are supplied for practising JSON layouts; they have not been retrieved by the API in this cell.


In [ ]:
example_df = pd.DataFrame({
    "Ligand": ["ATP", "NAD", "HEM"],
    "PDB_CCD": ["ATP", "NAD", "HEM"],
    "MolecularWeight": [507.18, 663.43, 616.49]
})

example_df


Then convert to JSON using `orient ="records"`

In [ ]:
records_json = example_df.to_json(
    orient="______",
    indent=2
)

print(______)


In the next two cells try `orient="columns"` and then `orient="index"`

In [ ]:
columns_json = ____.____(
    orient="_____",
    indent=____
)

print(______)


In [ ]:
index_json = ______._____(
    orient="______",
    indent=___
)

print(______)


**Suggested answer:** Records produces a top-level list of row dictionaries. Columns produces a dictionary keyed by column names; each value is keyed by row index. Index produces a dictionary keyed by row indices; each value is a row dictionary. Records most closely resembles a list of compound records, though a real API may wrap that list in further objects.

### Convert JSON records back into a DataFrame

Now reverse the process. Use `pd.read_json()` to reconstruct the DataFrame from the `"records"` JSON.

Because `records_json` contains JSON text rather than a filename, wrap it in `StringIO`.


In [ ]:


restored_df = ____._____(
    StringIO(______),
    orient="_____"
)

restored_df


The same table can therefore be represented using different valid JSON structures. This is useful when working with APIs because different services may organise their JSON responses differently.

You do not need to memorise every possible JSON structure: inspect the response first, then identify the dictionaries, lists and keys you need.


### Task 9 — Visualise a chemical-property relationship

Make a scatter plot of:

- molecular weight in units of g mol<sup>-1</sup> on the x-axis,
- Topological polar surface area in units of Å<sup>2</sup> on the y-axis.

Add the compound name next to each point using `plt.annotate()`.

Then consider whether the small set of compounds appears to show a simple relationship between molecular size and polar surface area.

In [ ]:
plt.figure(figsize=(7, 5))

plt.scatter(
    df["_____"],
    df["______"]
)

for _, row in df.iterrows():
    plt.annotate(
        row["______"],
        (row["______"], row["_____"])
    )

plt.xlabel("_____")
plt.ylabel("______")
plt.title("_____")

plt.tight_layout()
plt.show()


## Part 5 — Retrieving a chemical structure image

APIs do not have to return JSON. PubChem can also return a structure as a PNG image.

For example, the output format can be changed from `/JSON` to `/PNG`.

We can download the bytes using `urlopen()` and display the image directly inside the notebook.


### Task 10 — Display a molecular structure

The response here is PNG image bytes, not JSON. Complete the function using `quote(name, safe="")`, `urlopen(..., timeout=30)`, `.read()` and IPython's `Image`. No PIL conversion is needed. Test caffeine, pause 0.25 seconds, then try a molecule of your choice.

In [ ]:

def show_pubchem_structure(name):

    encoded_name = quote(____, ______)

    url = (
        "https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/name/"
        + encoded_name
        + "/PNG"
    )

    with urlopen(____, _____) as ________:
        image_bytes = response.read()

    display(
        Image(
            data=image_bytes,
            format="png"
        )
    )


show_pubchem_structure("_____")

# Another example:
time.sleep(0.25)
show_pubchem_structure("______")

## Part 6 — A more advanced search: chemical formulas

PUG REST can perform searches rather than just look up a known name or identifier.

A formula search asks for compounds matching a specified elemental composition. This is a useful reminder that a molecular formula does **not** uniquely specify chemical structure.

We will use PubChem's `fastformula` operation.


### Task 11 — Search by formula

Use the provided request to find up to 20 CIDs matching C8H10N4O2. Inspect the number returned and check whether the caffeine CID from Task 5 appears.

`MaxRecords=20` limits the returned list. Its length is not the total number of compounds with this formula. A molecule absent from this truncated list is not necessarily absent from PubChem. A formula specifies composition, not a unique structure.

In [ ]:
formula = "_______"

url = (
    "https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/fastformula/"
    + quote(formula, safe="")
    + "/cids/JSON?MaxRecords=20"
)

with urlopen(url, timeout=30) as response:
    text = response.read().decode("utf-8")

data = json.loads(_____)

cids = data["IdentifierList"]["____"]

print("Number returned:", len(cids))
print(cids)

print("Caffeine CID:", caffeine["____"])
print("Is caffeine in this result?", caffeine[_____] in ______)


### Task 12 — Function design challenge

Write a function:

```python
get_property(compound_name, property_name)
```

which retrieves **one requested PubChem property** for **one compound name**.

For example:

```python
get_property("acetone", "MolecularWeight")
```

should return the molecular weight.

Your function should:

- URL-encode the compound name,
- build the request URL from the two function arguments,
- parse JSON,
- return only the requested value,
- catch `HTTPError` and `URLError`,
- return `None` if the request fails.
- print some key properties of different compounds

Use documented current property names (for example, MolecularWeight, XLogP and TPSA). Try three combinations.

## Part 7 — GET versus POST

So far, all information required by the API has been placed in the URL. These are **GET** requests.

For more complicated searches, it is often cleaner to send a structured request body. This is commonly done using **POST**.

The RCSB PDB Search API accepts JSON query objects. We can therefore construct a Python dictionary, convert it to JSON, encode it as bytes, and send it using `urllib.request.Request`.


### Task 13 — Construct a PDB search query

The query below is designed to find:

- Protein structures determined by **X-ray diffraction**,
- with reported resolution **≤ 1.5 Å**.

Complete the missing pieces of the query dictionary.

Read the nesting carefully. API query objects often contain dictionaries inside lists inside dictionaries.

We retrieve at most 100 identifiers to keep the response small. The response also reports the total number of matches.

In [ ]:
pdb_query = {
    "query": {
        "type": "group",
        "logical_operator": "and",
        "nodes": [
            {
                "type": "terminal",
                "service": "text",
                "parameters": {
                    "attribute": "exptl.method",
                    "operator": "exact_match",
                    "value": "X-RAY DIFFRACTION"
                }
            },
            {
                "type": "terminal",
                "service": "text",
                "parameters": {
                    "attribute": "rcsb_entry_info.resolution_combined",
                    "operator": "less_or_equal",
                    "value": ____
                }
            }
        ]
    },
    "return_type": "entry",
    "request_options": {
        "paginate": {"start": 0, "rows": ____}
    }
}

print(json.dumps(_____, indent=2))


### Task 14 — Send a JSON POST request using urllib

To POST JSON using `urllib` we need to:

1. convert the Python dictionary to JSON using `json.dumps()`,
2. encode the JSON string to UTF-8 bytes,
3. create a `Request` object,
4. set the `Content-Type` header to `application/json`,
5. specify `method="POST"`,
6. open the request using `urlopen()`.

Complete the code and print the number of structures found.

In [ ]:
search_url = "https://search.rcsb.org/rcsbsearch/v2/query"

json_text = ____.____(_____)

request_data = json_text.encode("_____")

request = Request(
    search_url,
    data=request_data,
    headers={
        "Content-Type": "application/json"
    },
    method="_____"
)

with urlopen(request, timeout=30) as response:
    result_text = response.read().decode("utf-8")

results = json.loads(result_text) if result_text.strip() else {"total_count": 0, "result_set": []}

print("Total hits:", results["total_count"])


### Task 15 — Extract PDB identifiers

The response's `result_set` is a list of dictionaries. Extract each dictionary's `identifier` into a list called `pdb_ids`.

Print the first 20 IDs, the number retrieved, and the response's `total_count`. **Pagination** returns one portion of a larger result; we requested at most 100 entries. `total_count` reports all matches, while `len(pdb_ids)` reports this page.

Calculate the percentage of returned IDs beginning with a numeral. This is programming practice, not a chemical statistic. Handle an empty result to avoid division by zero.

In [ ]:
result_set = results.get("result_set", [])

pdb_ids = []

for item in result_set:
    pdb_ids.append(item["identifier"])

print(pdb_ids[:20])
#print number of IDs retrieved
print("IDs retrieved:", _____(______))
print("Total matching entries:", results["total_count"])

starts_with_number = [
    pdb_id for pdb_id in pdb_ids
    if pdb_id[0].isdigit()
]

percentage = (
    len(_______)
    / len(_____)
    * 100
) if pdb_ids else float("nan")

print("Percentage beginning with a numeral:", percentage)


## Part 8 — Structural chemistry: cofactors in the PDB

Rather than merely retrieving an arbitrary PDB entry, we will investigate a chemical question:

> **How are common biological cofactors represented in high-resolution protein structures?**

We will work with ATP, ADP, NAD, FAD and haem. These are chemically diverse molecules that occur in many protein structures.

The PDB identifies small molecules using **Chemical Component Dictionary (CCD) codes**. For example, `HEM` is the PDB component code for haem.

The PDB and PubChem do **not** use the same identifiers. That makes this a useful example of a real data-integration problem.


### Task 16 — Search the PDB for a ligand under chemically useful constraints

Define a function `search_pdb_ligand(ligand_code, resolution=2.0)`.

Require:

- the specified ligand,
- **X-RAY DIFFRACTION**,
- resolution **2.0 Å or better**.

The three conditions are represented as dictionaries inside the `"nodes"` JSON array.

In [ ]:
def ____________________________:

    query = {
        "query": {
            "type": "group",
            "logical_operator": "and",
            "nodes": [
                {
                    "type": "terminal",
                    "service": "text",
                    "parameters": {
                        "attribute": "rcsb_nonpolymer_entity_container_identifiers.nonpolymer_comp_id",
                        "operator": "exact_match",
                        "value": ligand_code
                    }
                },
                {
                    "type": "terminal",
                    "service": "text",
                    "parameters": {
                        "attribute": "exptl.method",
                        "operator": "exact_match",
                        "value": "_________"
                    }
                },
                {
                    "type": "terminal",
                    "service": "text",
                    "parameters": {
                        "attribute": "rcsb_entry_info.resolution_combined",
                        "operator": "less_or_equal",
                        "value": ______
                    }
                }
            ]
        },
        "return_type": "entry",
        "request_options": {"paginate": {"start": 0, "rows": 100}}
    }

    # Python dictionary -> JSON string -> bytes
    request_data = json.dumps(query).encode("utf-8")

    request = Request(
        "https://search.rcsb.org/rcsbsearch/v2/query",
        data=request_data,
        headers={"Content-Type": "application/json"},
        method="POST"
    )

    with urlopen(request, timeout=30) as response:
        response_text = response.read().decode("utf-8")

    # JSON string -> Python dictionary
    return json.loads(response_text) if response_text.strip() else {"total_count": 0, "result_set": []}


atp_results = search_pdb_ligand("_____")

print("ATP-containing structures:", atp_results["total_count"])
print(atp_results.keys())


### Task 17 — Inspect the returned JSON

Before extracting values from an unfamiliar API response, inspect it.

Print the first object in `"result_set"` using `json.dumps(..., indent=2)`, then make a list containing the first 10 PDB identifiers.

Why is inspecting the JSON first safer than guessing its structure?

In [ ]:
print(
    _____.________(
        (atp_results.get("result_set", []) or [{}])[0],
        indent=2
    )
)

atp_ids = [
    item["identifier"]
    for item in atp_results.get("result_set", [])
]

print(______[_____])


**Suggested answer:** Inspecting the response reveals the actual keys, lists and optional fields. Guessing can select the wrong value or fail when the response changes or contains an error message.

## Part 9 — Combine data from two APIs

PubChem provides chemical properties; RCSB provides counts of matching PDB entries. We will create a table from each service and merge them.

A **merge** matches rows using a shared column. Here `PDB_CCD` is a mapping we assign ourselves, not proof that PubChem and the PDB describe precisely the same chemical species. Name searches can select different protonation, oxidation or salt forms—particularly for haem and nucleotide cofactors. Inspect CIDs, formulas and structures before a scientific comparison. This workshop demonstrates a tentative mapping, not a validated identity match.

**Workflow:** request → inspect JSON → make tables → check types → merge → plot.

### Task 18 — Build a PubChem DataFrame

So far, we have retrieved chemical information for individual molecules using the PubChem API. We can now automate this process to retrieve information for several biologically important cofactors and organise the results into a single pandas DataFrame.

1. Create a list of cofactors containing the compound name used to search PubChem and its corresponding Protein Data Bank (PDB) Chemical Component Dictionary code. Each entry is a tuple, containing two related values.

2. Use a for loop to work through the list, passing each compound name to the safe_pubchem_properties() function. This retrieves the molecular properties from PubChem.

3. Organise the retrieved information into a dictionary called record, using descriptive column names such as MolecularWeight, Formula and TPSA. Add each dictionary to the pubchem_records list using .append(). The if properties is not None statement ensures that only successful API requests are included.

4. Include a short delay using time.sleep(0.25) between requests to avoid sending too many requests to the PubChem server in quick succession.

5. Convert the collected records into a pandas DataFrame using pd.DataFrame(). Display the first five rows using .head(), then inspect the column names and data types using .columns and .dtypes.

6. Sort the compounds by molecular weight.

In [ ]:
cofactors = [
    ("ATP", "ATP"),
    ("ADP", "ADP"),
    ("NAD", "NAD"),
    ("FAD", "FAD"),
    ("heme", "HEM")
]

pubchem_records = []

for ________________:

    properties = _______(______)

    if properties is not None:

        record = {
            "Name": name,
            "PDB_CCD": pdb_code,
            "PubChem_CID": properties["CID"],
            "Formula": properties["MolecularFormula"],
            "MolecularWeight": properties["MolecularWeight"],
            "XLogP": properties.get("XLogP"),
            "HBondDonors": properties["HBondDonorCount"],
            "HBondAcceptors": properties["HBondAcceptorCount"],
            "TPSA": properties["TPSA"]
        }

        _______._____(_____)

    time.sleep(0.25)

pubchem_df = ____._______(pubchem_records)

display(_____.head())

print(pubchem_df.columns)
print(pubchem_df.dtypes)



# Convert before sorting; API numbers may arrive as text.
pubchem_df["MolecularWeight"] = -____._____(pubchem_df["_______"], errors="________")
#sort values
display(pubchem_df._________("_________"))

### Task 20 — Build a PDB table

For each cofactor CCD code, retrieve the number of X-ray entries with resolution ≤ 2.0 Å. Store the code and `total_count` in `pdb_records`, then make `pdb_df`.

The search returns at most 100 identifiers but `total_count` includes all matching entries. These are **entry counts**, not ligand copy counts or independent protein counts. A successful no-match response is zero; a request failure must not be recorded as zero.

In [ ]:
pdb_records = []

for name, pdb_code in cofactors:

    results = search_pdb_ligand(
        pdb_code,
        resolution=___________
    )

    _______.________({
        "PDB_CCD": pdb_code,
        "PDB_structures_2A": results["________"]
    })

    time.sleep(0.25)

pdb_df = _______._______(_______)

pdb_df


### Task 21 — Merge data from the two APIs

Both DataFrames contain a column called `"PDB_CCD"`.

Use `pd.merge()` to join the PubChem and PDB datasets using this common identifier.

Use an **inner join**, meaning that only rows with a matching identifier in both datasets will be retained.

Then display the merged DataFrame.

This is the key aggregation step: data retrieved independently from two APIs have become one analysable dataset.

In [ ]:
combined_df = _____.______(
    pubchem_df,
    pdb_df,
    on="PDB_CCD",
    how="inner"
)

combined_df


### Task 22 — Check data types, then filter and sort

Before carrying out numerical comparisons, check how pandas has interpreted each column. Data returned by an API may sometimes be stored as text even when it represents a number.

In [ ]:
print(______._______)

Convert the columns that we want to treat as numerical data (MolecularWeight, TPSA and PDB_structures_2A) using `pd.to_numeric()`:

In [ ]:
combined_df["________"] = ____._______(
    combined_df["__________"]
)

combined_df["________"] = ____._____(
    combined_df[""]
)

combined_df[""] = ____.________(
    combined_df[""]
)

#print the data types in combined_df
print(_______._______)

Now select cofactors with a molecular weight greater than 500 g mol⁻¹:

In [ ]:
large = combined_df[
    combined_df["_______"] > _____
]

display(large)

Sort the compounds from largest to smallest TPSA:

In [ ]:
display(
    combined_df._______(
        by="________",
        ascending=_______
    )
)

Then sort by the number of high-resolution PDB structures:

In [ ]:
display(
    combined_df._______(
        by="________",
        ascending=_______
    )
)

**Suggested answer:** Strings contain text rather than numeric values; comparing them with integer 500 raises a type error. Sorting strings can also give lexical rather than numeric order. Convert with pd.to_numeric before numerical analysis.

### Task 23 — Calculate a new column

We can now safely use the numerical columns in calculations. Calculate the number of high-resolution PDB structures per 100 Da of molecular weight:

In [ ]:
combined_df["PDB_structures_per_100_Da"] = (
    combined_df[____]
    / combined_df[____]
    * 100
)

display(_____)

This calculated quantity is **not intended to be a chemically fundamental property**. It simply demonstrates how pandas can perform a calculation across every row of a DataFrame without writing a loop.

### Task 24 — Visualise the aggregated data

In the code cell below plot molecular weight against the number of high-resolution PDB structures:

### Extension — `groupby()` and larger datasets

With only five cofactors there is little need for grouping, but larger API-derived datasets often contain repeated categories.

For example, if your table contained an additional `"CofactorClass"` column, you could calculate a mean for each class using:

```python
combined_df.groupby("CofactorClass")["MolecularWeight"].mean()
```

In the independent challenge, consider whether your chosen dataset contains a categorical variable that would make `groupby()` scientifically useful.

At this point you have used pandas for:

- `DataFrame`
- `.head()`
- `.columns`
- `.dtypes`
- `.sort_values()`
- boolean filtering
- `pd.merge()`
- creation of calculated columns
- preparation of data for plotting

The APIs retrieve the data; **pandas turns the responses into a dataset that can be analysed**.


## Part 10 — Independent challenge

### Task 25 — Build a mini chemical-data pipeline

Choose 5–10 chemically related compounds. Retrieve at least four PubChem properties, handle failed requests, pause between requests, make a DataFrame, save a CSV and plot one chemical relationship.

Record the query names, CIDs, endpoint and retrieval time. Explain the observed trend and one limitation. Suggested series: alcohols, substituted benzenes or common NSAIDs. Save only to a filename you choose for this workshop.

Use the code cell below:

## Summary

In this workshop you have used Python to:

- construct API URLs,
- safely encode chemical names,
- send HTTP GET requests with `urllib`,
- decode bytes returned over the web,
- parse JSON into Python dictionaries and lists,
- extract chemical properties from nested data,
- handle HTTP and network errors,
- respect API request-rate limits,
- automate queries over sets of compounds,
- convert API results into tabular data,
- visualise chemical-property relationships,
- retrieve non-JSON data such as molecular structure images,
- perform chemical formula searches,
- construct and submit a JSON POST request,
- query structural data from the Protein Data Bank.

The important idea is that an API is not merely a convenient source of numbers: it is a reproducible interface between your analysis code and a scientific database.


***Don't forget to save your notebook and submit to minerva***
